# 26 — Training the ABSA model (Stage 1: English, 6 aspects)

A **teaching walkthrough** of training the aspect-based sentiment model on the
**human-annotated English** reviews. Every step has an explanation.

### What we're building

**Joint ABSA as multi-head classification.** For one review we predict, *at the same
time*, a sentiment for **each of the 6 macro aspects**:

> `Facility · Amenity · Service · Experience · Loyalty · Branding`

Each aspect gets one of **4 labels**: `not_mentioned / negative / neutral / positive`.

```
                                   ┌─► head: Facility   → 4 classes
                                   ├─► head: Amenity    → 4 classes
  review text ─► [ transformer ]──┼─► head: Service    → 4 classes
                  (shared encoder)├─► head: Experience → 4 classes
                                   ├─► head: Loyalty    → 4 classes
                                   └─► head: Branding   → 4 classes
```

One shared encoder learns the language; six small linear heads each specialise in one
aspect. The loss is the **sum** of the six heads' losses, so a single backward pass
trains everything jointly — that's the "joint" in joint ABSA.

### Why this setup

- **English-only, monolingual encoder** (`distilbert-base-uncased`) for now — small,
  fast, easy to reason about. Once you understand this, swapping in a multilingual
  encoder (`xlm-roberta-base`) and adding the Vietnamese annotations is a 2-line change
  (see the last section).
- **6 aspects, not 5** — we train on the annotators' native label set, which includes
  `Branding`. (The 5-aspect `other`-folded scheme was only for the LLM silver labels.)
- **Trained purely on human gold** (`Annoted_TripAdvisor_EN.json`) — no LLM labels here.

In [1]:
import sys, warnings, json
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, f1_score, confusion_matrix

from absa_en_data import (
    ASPECTS, CLASSES, ID2CLASS,
    load_docs, split_docs, class_weights, label_distribution,
)

# ---- config (tune these) --------------------------------------------------
ENCODER   = "distilbert-base-uncased"  # English. Swap to xlm-roberta-base for bilingual
MAX_LEN   = 128     # tokens per review (reviews avg ~450 chars ~ 90 tokens)
BATCH     = 16
EPOCHS    = 3
LR        = 2e-5
MAX_TRAIN = 3000    # cap train docs for a CPU-friendly run; None = use all ~7.5k
FINE_TUNE = True    # True = fine-tune the encoder; False = frozen baseline (faster)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| encoder:", ENCODER)
print("aspects:", ASPECTS)
print("classes:", CLASSES)

device: cpu | encoder: distilbert-base-uncased
aspects: ['Facility', 'Amenity', 'Service', 'Experience', 'Loyalty', 'Branding']
classes: ['not_mentioned', 'negative', 'neutral', 'positive']


## Step 1 — From span annotations to document labels

The annotators highlighted **text spans** and tagged each with an aspect + a sentiment
(Label Studio format). But our model classifies a **whole review**, so we must reduce
each document's spans to **one sentiment per aspect**:

- aspect has ≥1 span → **majority** sentiment over those spans (ties → `neutral`, i.e. a
  genuinely mixed aspect like *"room clean but bathroom dirty"*)
- aspect has no span → `not_mentioned`

`load_docs()` (in `src/absa_en_data.py`) does this. Let's look at one example so the
transformation is concrete.

In [2]:
from absa_en_data import doc_to_labels, EN_SOURCES

raw = json.loads(EN_SOURCES["tripadvisor"].read_text(encoding="utf-8"))
text, labels = doc_to_labels(raw[0])
print("REVIEW:\n ", text[:400], "...\n")
print("AGGREGATED LABELS (one per aspect):")
for a, s in labels.items():
    print(f"  {a:11} -> {s}")

REVIEW:
  We enjoyed our stay here, the hotel is as you would expect from the Pullman brand. Our room was clean and the beds were so comfortable. Check in and out easy. We ate at the restaurant and my only complaint is our wait staff were very slow, slow to hand out menus then very slow to take our orders. Overall though we are happy we stayed here. ...

AGGREGATED LABELS (one per aspect):
  Facility    -> positive
  Amenity     -> not_mentioned
  Service     -> neutral
  Experience  -> positive
  Loyalty     -> not_mentioned
  Branding    -> not_mentioned


In [3]:
# Load all English docs as (text, [class_id per aspect]) and inspect the label balance.
rows = load_docs(["tripadvisor"])   # add "booking" for +7k docs (also English)
print(f"total labeled documents: {len(rows):,}\n")

import pandas as pd
dist = label_distribution(rows)
pd.DataFrame({a: dist[a] for a in ASPECTS}).T[CLASSES].fillna(0).astype(int)

total labeled documents: 9,990



,not_mentioned,negative,neutral,positive
Facility,2535,527,958,5970
Amenity,4345,191,476,4978
Service,865,459,625,8041
Experience,3969,476,448,5097
Loyalty,6026,190,80,3694
Branding,8784,197,40,969


Notice the **imbalance**: most aspects are dominated by `not_mentioned` and `positive`;
`negative`/`neutral` are rare, and `Branding` is barely mentioned at all. A naive model
would just predict the majority class and score ~0 on the classes we actually care about.
We fix that with **class weights** in Step 6.

## Step 2 — Train / validation / test split

- **train** — the model learns from this
- **val** — checked after each epoch to pick the best model (not learned from)
- **test** — touched **once** at the very end for the honest score

The split is a fixed random seed, so it's reproducible.

In [4]:
splits = split_docs(rows, val_frac=0.1, test_frac=0.15, seed=42)
if MAX_TRAIN:
    splits["train"] = splits["train"][:MAX_TRAIN]
print({k: len(v) for k, v in splits.items()})

{'train': 3000, 'val': 999, 'test': 1498}


## Step 3 — Tokenization

A transformer can't read text — it reads **token ids**. The tokenizer splits the review
into sub-word pieces and maps them to integers, adds the special `[CLS]`/`[SEP]` tokens,
and pads/truncates to `MAX_LEN`. It also returns an **attention mask** (1 = real token,
0 = padding) so the model ignores padding.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER)

demo = tokenizer("The room was clean but the staff were rude.",
                 truncation=True, max_length=MAX_LEN)
print("tokens   :", tokenizer.convert_ids_to_tokens(demo["input_ids"]))
print("input_ids:", demo["input_ids"])
print("attention:", demo["attention_mask"])

tokens   : ['[CLS]', 'the', 'room', 'was', 'clean', 'but', 'the', 'staff', 'were', 'rude', '.', '[SEP]']
input_ids: [101, 1996, 2282, 2001, 4550, 2021, 1996, 3095, 2020, 12726, 1012, 102]
attention: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Step 4 — A PyTorch `Dataset`

Wraps `(text, labels)` so the `DataLoader` can hand the model batches of
`(input_ids, attention_mask, labels)`. `labels` is a length-6 vector of class ids
(one per aspect).

In [6]:
class AspectDataset(Dataset):
    def __init__(self, rows, tok, max_len):
        self.rows, self.tok, self.max_len = rows, tok, max_len
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        text, labels = self.rows[i]
        enc = self.tok(text, truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        return enc["input_ids"][0], enc["attention_mask"][0], torch.tensor(labels)

train_dl = DataLoader(AspectDataset(splits["train"], tokenizer, MAX_LEN), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(AspectDataset(splits["val"],   tokenizer, MAX_LEN), batch_size=BATCH)
test_dl  = DataLoader(AspectDataset(splits["test"],  tokenizer, MAX_LEN), batch_size=BATCH)

ids, mask, y = next(iter(train_dl))
print("batch input_ids :", ids.shape)      # (BATCH, MAX_LEN)
print("batch labels    :", y.shape)        # (BATCH, 6 aspects)
print("first row labels:", y[0].tolist(), "->", [ID2CLASS[c] for c in y[0].tolist()])

batch input_ids : torch.Size([16, 128])
batch labels    : torch.Size([16, 6])
first row labels: [0, 0, 1, 0, 0, 0] -> ['not_mentioned', 'not_mentioned', 'negative', 'not_mentioned', 'not_mentioned', 'not_mentioned']


## Step 5 — The model: shared encoder + 6 heads

1. The encoder turns the token ids into a **contextual vector**. We take the vector at
   the `[CLS]` position (index 0) as the review's summary embedding.
2. Each of the 6 **heads** is a tiny `Linear(hidden → 4)` that turns that embedding into
   4 class scores for its aspect.

All heads share the same encoder, so improving the encoder for one aspect helps them all.

In [7]:
class MultiAspectClassifier(nn.Module):
    def __init__(self, encoder_name, n_aspects, n_classes):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleList([nn.Linear(hidden, n_classes) for _ in range(n_aspects)])
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0])   # [CLS] embedding
        return [head(cls) for head in self.heads]         # list of (B, 4) logits

model = MultiAspectClassifier(ENCODER, len(ASPECTS), len(CLASSES)).to(device)
if not FINE_TUNE:
    for p in model.encoder.parameters():
        p.requires_grad = False
    print("encoder FROZEN — only the 6 heads train (fast baseline)")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params/1e6:.1f}M")

# one forward pass to see the output shape
with torch.no_grad():
    demo_logits = model(ids.to(device), mask.to(device))
print(f"model returns {len(demo_logits)} logit tensors (one per aspect), each {tuple(demo_logits[0].shape)}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable parameters: 66.4M


model returns 6 logit tensors (one per aspect), each (16, 4)


## Step 6 — Class weights (handling imbalance)

Because `positive`/`not_mentioned` dominate, we weight each class **inversely to its
frequency** so a mistake on a rare `negative` costs the model more than a mistake on a
common `positive`. One weight vector per aspect (they have different balances).

In [8]:
weights = class_weights(splits["train"])
losses = [nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float).to(device))
          for w in weights]

pd.DataFrame(weights, index=ASPECTS, columns=CLASSES).round(2)

,not_mentioned,negative,neutral,positive
Facility,0.96,4.49,2.67,0.42
Amenity,0.56,17.05,5.03,0.51
Service,3.16,5.77,4.01,0.31
Experience,0.63,5.56,5.21,0.49
Loyalty,0.41,15.96,28.85,0.68
Branding,0.28,14.71,62.50,2.62


## Step 7 — The training loop

The core of the whole notebook. Each step:
1. **forward** the batch → 6 logit tensors
2. compute each head's cross-entropy loss and **sum** them → the joint loss
3. **backward** — gradients flow into every head *and* the shared encoder
4. **step** — the optimizer nudges the weights

After each epoch we score the **val** set and keep the best model.

In [9]:
def macro_f1(model, loader):
    """Mean over aspects of the per-aspect macro-F1."""
    model.eval()
    preds = {a: [] for a in range(len(ASPECTS))}
    gold  = {a: [] for a in range(len(ASPECTS))}
    with torch.no_grad():
        for ids, mask, yb in loader:
            logits = model(ids.to(device), mask.to(device))
            for a in range(len(ASPECTS)):
                preds[a] += logits[a].argmax(1).cpu().tolist()
                gold[a]  += yb[:, a].tolist()
    f1s = {ASPECTS[a]: f1_score(gold[a], preds[a], average="macro", zero_division=0)
           for a in range(len(ASPECTS))}
    return sum(f1s.values()) / len(f1s), f1s, preds, gold

opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

best_f1, best_state = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, (ids, mask, yb) in enumerate(train_dl):
        ids, mask, yb = ids.to(device), mask.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(ids, mask)
        loss = sum(losses[a](logits[a], yb[:, a]) for a in range(len(ASPECTS)))
        loss.backward()
        opt.step()
        running += loss.item()
        if step % 50 == 0:
            print(f"  epoch {epoch} step {step:4d}/{len(train_dl)}  loss={loss.item():.3f}")
    val_f1, _, _, _ = macro_f1(model, val_dl)
    print(f"epoch {epoch}: train_loss={running/len(train_dl):.3f}  val_macroF1={val_f1:.3f}\n")
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"best val macro-F1: {best_f1:.3f}")

  epoch 1 step    0/188  loss=8.642


  epoch 1 step   50/188  loss=9.021


  epoch 1 step  100/188  loss=7.715


  epoch 1 step  150/188  loss=6.951


epoch 1: train_loss=6.851  val_macroF1=0.439



...

  epoch 3 step    0/188  loss=4.873


  epoch 3 step   50/188  loss=3.240


  epoch 3 step  100/188  loss=3.613


  epoch 3 step  150/188  loss=6.602


epoch 3: train_loss=4.593  val_macroF1=0.464

best val macro-F1: 0.467


## Step 8 — Evaluate on the held-out test set

**Macro-F1** averages the F1 of all 4 classes equally, so it doesn't let the model "win"
by only getting the common classes right — it's the honest metric for imbalanced data.
We report it per aspect, then drill into one aspect's per-class scores and confusion.

In [10]:
test_macro, per_aspect, preds, gold = macro_f1(model, test_dl)
print(f"TEST macro-F1 (mean over aspects): {test_macro:.3f}\n")
pd.Series(per_aspect, name="macro_F1").round(3).to_frame()

TEST macro-F1 (mean over aspects): 0.453



,macro_F1
Facility,0.497
Amenity,0.408
Service,0.507
Experience,0.436
Loyalty,0.492
Branding,0.380


In [11]:
# Per-class breakdown for one aspect (change the name to inspect others)
a = ASPECTS.index("Service")
print("=== Service — per-class ===")
print(classification_report(gold[a], preds[a], labels=list(range(len(CLASSES))),
                            target_names=CLASSES, zero_division=0))
print("confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(gold[a], preds[a], labels=list(range(len(CLASSES)))),
                   index=CLASSES, columns=CLASSES))

=== Service — per-class ===
               precision    recall  f1-score   support

not_mentioned       0.33      0.42      0.37       139
     negative       0.44      0.70      0.54        64
      neutral       0.20      0.52      0.29        96
     positive       0.93      0.75      0.83      1199

     accuracy                           0.70      1498
    macro avg       0.47      0.60      0.51      1498
 weighted avg       0.81      0.70      0.74      1498

confusion matrix (rows=true, cols=pred):
               not_mentioned  negative  neutral  positive
not_mentioned             58        18       22        41
negative                   3        45       13         3
neutral                    5        16       50        25
positive                 108        24      166       901


## Step 9 — Try it on new text

The real test of understanding: feed the model a fresh sentence and read off all 6
aspect predictions at once.

In [12]:
def predict(text):
    model.eval()
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    padding="max_length", return_tensors="pt")
    with torch.no_grad():
        logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
    out = {}
    for a, aspect in enumerate(ASPECTS):
        cls = logits[a].argmax(1).item()
        if ID2CLASS[cls] != "not_mentioned":
            out[aspect] = ID2CLASS[cls]
    return out

for t in [
    "The beachfront location was stunning but the room smelled of damp and the AC was broken.",
    "Staff went above and beyond, we will definitely be back!",
    "Overpriced for what you get.",
]:
    print(t)
    print("  ->", predict(t), "\n")

The beachfront location was stunning but the room smelled of damp and the AC was broken.
  -> {'Facility': 'negative', 'Amenity': 'negative', 'Service': 'negative', 'Experience': 'negative', 'Branding': 'negative'} 

Staff went above and beyond, we will definitely be back!
  -> {'Service': 'positive', 'Loyalty': 'positive'} 

Overpriced for what you get.
  -> {'Facility': 'negative', 'Service': 'negative', 'Experience': 'negative', 'Loyalty': 'negative', 'Branding': 'negative'} 



## Step 10 — Save the model

In [13]:
from pathlib import Path
MODEL_DIR = Path("../models/absa_en")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), MODEL_DIR / "model.pt")
tokenizer.save_pretrained(MODEL_DIR)
(MODEL_DIR / "label_config.json").write_text(
    json.dumps({"aspects": ASPECTS, "classes": CLASSES, "encoder": ENCODER}, indent=2))
print("saved ->", MODEL_DIR.resolve())

saved -> C:\Users\darkn\coastal-hotel\models\absa_en


## Recap + how to scale up

You just trained a joint 6-aspect × 4-class sentiment model:
**span annotations → document labels → tokenize → shared encoder + 6 heads →
class-weighted joint loss → macro-F1 evaluation → inference.**

**To improve results** (this notebook used a CPU-friendly subset):
- set `MAX_TRAIN = None` (use all docs) and `EPOCHS = 4`
- add the second English source: `load_docs(["tripadvisor", "booking"])` (+7k docs, more negatives)
- run on a GPU — the exact same code, `device` auto-switches to `cuda`

**Headless / reproducible run** (same logic, no notebook):
```bash
uv run python src/absa_train_en.py --epochs 3                       # subset off = all docs
uv run python src/absa_train_en.py --sources tripadvisor booking    # both EN sources
uv run python src/absa_train_en.py --no-fine-tune                   # frozen baseline
```

### Next: add Vietnamese (bilingual model)

When your Vietnamese annotations are ready, this becomes bilingual with **two changes**:
1. `ENCODER = "xlm-roberta-base"` — a multilingual encoder that understands vi + en.
2. Extend `EN_SOURCES` / `load_docs` in `src/absa_en_data.py` to also read the Vietnamese
   Label Studio export, and load both: `load_docs(["tripadvisor", "booking", "vi_xxx"])`.

Everything else — the Dataset, the model, the loss, the loop, the metrics — stays exactly
the same. That's the payoff of getting the English pipeline right first.